# Notebook of the results section

**All plots in the report can be reproduced here**

To save the plots as pgf figures, you will need a working latex distribution
with those apt packages installed

```bash
sudo apt install dvipng cm-super texlive-fonts-recommended
!sudo apt-get install texlive-latex-recommended
!sudo apt install texlive-latex-extra
!sudo apt install texlive-xetex
!sudo apt install ttf-mscorefonts-installer
```


In [ ]:
from __future__ import annotations
from pathlib import Path

from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from sklearn.decomposition import PCA
from wandb.apis.public import Run
from fm_benchmark_remote_sensing.data import PASTIS_LABEL_NAMES


import wandb
import os
import pandas as pd
import scienceplots  # noqa: F401
from typing import cast
import json
import seaborn as sns
from matplotlib.colors import LogNorm
import geopandas as gpd
from fm_benchmark_remote_sensing.data import (
    PastisEmbeddingPreviewDataset,
)
import contextily as cx

api = wandb.Api()


In [ ]:
plt.style.use(["science", "grid", "ieee", "pgf"])
plt.rcParams.update(
    {
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
        "pgf.preamble": "\n".join(
            [
                r"\usepackage{fontspec}",
                r"\usepackage{unicode-math}",
                r"\setmainfont{Times New Roman}",
                r"\usepackage{siunitx}",
            ]
        ),
    }
)


In [ ]:
wandb.login()


In [ ]:
output_dir = "../report/figures"
os.makedirs(output_dir, exist_ok=True)
GOLDEN_RATIO = (9, 5)

In [ ]:
dataset_artifact = api.artifact(
    "maxerbox-org/benchmark-fm-models-remote-sensing/preview-pertuis-pastis-r-embeddings:latest",
    type="dataset",
)

preview_root = Path("../data/preview-pertuis-pastis-r-embeddings")
if not preview_root.exists():
    dataset_artifact.download(root=preview_root)

dataset = PastisEmbeddingPreviewDataset(
    pastis_r_root=preview_root / "PASTIS_R",
    ae_root=preview_root / "AE_EMBEDDING",
    tessera_root=preview_root / "TESSERA_EMBEDDING",
)


In [ ]:
## Train runs

train_alpha_earth_ruche_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/a0tm2lew"
)
train_tessera_ruche_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/j090rbd2"
)
alise_ruche_train_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/02za7us6"
)

## Test preview runs

test_preview_alpha_earth_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/p4k1faso"
)
test_preview_tessera_h512_h256_lr0: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/syoboq46"
)
test_preview_alise_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/gc6uhmw2"
)

## Test runs

test_alpha_earth_ruche_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/o7pp2det"
)
test_tessera_ruche_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/8nzfuu94"
)
test_alise_ruche_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/kqujylye"
)

## Test runs scarce data

test_alpha_earth_ruche_scarce_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/d1abbboi"
)
test_tessera_ruche_scarce_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/75nig2j6"
)
test_alise_ruche_scarce_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/sprogbbv"
)

## Test preview runs scarce data

test_preview_alpha_earth_scarce_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/zpt956za"
)
test_preview_tessera_scarce_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/odqqhjdq"
)
test_preview_alise_scarce_h512_h256: Run = api.run(
    "maxerbox-org/benchmark-fm-models-remote-sensing/d6p9yku6"
)
train_runs: list[Run] = [
    train_alpha_earth_ruche_h512_h256,
    train_tessera_ruche_h512_h256,
    alise_ruche_train_h512_h256,
]
test_runs: list[Run] = [
    test_alpha_earth_ruche_h512_h256,
    test_tessera_ruche_h512_h256,
    test_alise_ruche_h512_h256,
]
test_runs_scarce: list[Run] = [
    test_alpha_earth_ruche_scarce_h512_h256,
    test_tessera_ruche_scarce_h512_h256,
    test_alise_ruche_scarce_h512_h256,
]
preview_runs: list[Run] = [
    test_preview_alpha_earth_h512_h256,
    test_preview_tessera_h512_h256_lr0,
    test_preview_alise_h512_h256,
]
preview_runs_scarce: list[Run] = [
    test_preview_alpha_earth_scarce_h512_h256,
    test_preview_tessera_scarce_h512_h256,
    test_preview_alise_scarce_h512_h256,
]

### Metrics over training epochs for the validation fold


In [ ]:
def plot_val_train_metric(*runs: Run):
    # ---- mIoU plot ----
    _, ax_miou = plt.subplots(figsize=GOLDEN_RATIO)

    for run in runs:
        name = run.name
        history_df: pd.DataFrame = cast(pd.DataFrame, run.history())
        eval_data = history_df[["epoch", "val/mIoU_epoch"]].dropna()

        ax_miou.plot(
            eval_data["epoch"],
            eval_data["val/mIoU_epoch"],
            marker="o",
            markersize=4,
            linewidth=1.5,
            label=name,
        )

    ax_miou.set_xlabel("Epoch")
    ax_miou.set_ylabel("mIoU")
    ax_miou.set_title("Validation mIoU over training epochs")
    ax_miou.grid(True, alpha=0.3)
    ax_miou.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "val_train_miou_all_runs.pdf"), backend="pgf")
    plt.show()

    # ---- F1 Macro plot ----
    _, ax_f1 = plt.subplots(figsize=GOLDEN_RATIO)

    for run in runs:
        name = run.name
        history_df: pd.DataFrame = cast(pd.DataFrame, run.history())
        eval_data = history_df[["epoch", "val/F1_macro_epoch"]].dropna()

        ax_f1.plot(
            eval_data["epoch"],
            eval_data["val/F1_macro_epoch"],
            marker="o",
            markersize=4,
            linewidth=1.5,
            label=name,
        )

    ax_f1.set_xlabel("Epoch")
    ax_f1.set_ylabel("F1 Macro")
    ax_f1.set_title("Validation F1 Macro over training epochs")
    ax_f1.grid(True, alpha=0.3)
    ax_f1.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "val_train_f1_all_runs.pdf"), backend="pgf")
    plt.show()


In [ ]:
plot_val_train_metric(*train_runs)


### Summary metrics (F1 macro, IoU macro, etc.) for the whole test fold


In [ ]:
def export_test_run_metrics_to_latex(
    *runs: Run,
    caption: str = "Test metrics",
    label: str = "tab:test_metrics",
    output_filename: str = "test_metrics_table.tex",
):
    """Export test metrics to LaTeX table with bold formatting for highest values."""
    metrics = {}
    for run in runs:
        run_metrics = run.summary
        metrics[run.name] = {
            "mIoU": run_metrics.get("test/mIoU_epoch", "N/A"),
            "F1 Macro": run_metrics.get("test/F1_macro_epoch", "N/A"),
        }
    df = pd.DataFrame(metrics).T

    # Format with bold for max values in each column
    def format_with_bold(df_to_format: pd.DataFrame) -> str:
        latex_str = df_to_format.to_latex(
            float_format="%.4f",
            caption=caption,
            label=label,
            escape=False,  # Don't escape special characters
        )

        # Find max value for each metric column and bold it
        for col in df_to_format.columns:
            # Skip non-numeric columns
            if df_to_format[col].dtype not in [np.float64, np.int64, float, int]:
                continue

            max_val = df_to_format[col].max()
            if pd.notna(max_val):
                # Format the max value
                max_str = f"{max_val:.4f}"
                bold_max_str = r"\textbf{" + max_str + "}"
                # Replace the first occurrence of the max value in the table
                latex_str = latex_str.replace(max_str, bold_max_str, 1)

        return latex_str

    latex_output = format_with_bold(df)

    # Write to file
    with open(os.path.join(output_dir, output_filename), "w") as f:
        f.write(latex_output)

In [ ]:
export_test_run_metrics_to_latex(*test_runs)

### Summary metrics for scarce data (30 patches per fold)


In [ ]:
export_test_run_metrics_to_latex(
    *test_runs_scarce,
    caption="Test metrics with scarce data (30 patches per fold)",
    label="tab:test_metrics_scarce",
    output_filename="test_metrics_scarce_table.tex",
)

### Confusion matrix


In [ ]:
def export_confusion_matrix_test_runs(*runs: Run):
    for run in runs:
        name = run.name
        confmat_path = run.summary["confmat_test_table"]["path"]
        table = json.load(run.file(confmat_path).download(exist_ok=True))
        df = pd.DataFrame(table["data"], columns=table["columns"])
        df["nPredictions"] = df["nPredictions"].astype(int)
        # Colums = Actual, predicted, nPredictions

        cm = df.pivot(
            index="Actual", columns="Predicted", values="nPredictions"
        ).fillna(0)
        fig, ax = plt.subplots(figsize=(18, 16))
        cm_safe = cm.replace(0, 1)
        sns.heatmap(
            cm_safe,
            cmap="viridis",
            annot=cm,
            fmt="d",
            ax=ax,
            norm=LogNorm(vmin=1, vmax=cm.values.max()),  # log scale
            cbar_kws={"label": "nPredictions (log scale)"},
        )

        ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

        ax.set_title("Confusion Matrix (log scale)")
        plt.tight_layout()
        plt.show()
        fig.savefig(
            os.path.join(output_dir, f"confusion_matrix_{name}.pdf"), backend="pgf"
        )

In [ ]:
export_confusion_matrix_test_runs(*test_runs)

In [ ]:
@dataclass(frozen=True)
class _PredictionPayload:
    patch_id: int
    embedding_hwd: np.ndarray  # (H,W,D)
    ground_truth_hw: np.ndarray  # (H,W) in 0..17 (+ ignore) OR 0..19 (PASTIS)
    prediction_hw: np.ndarray  # (H,W) in 0..17 OR 1..18
    ignore_mask_hw: np.ndarray  # (H,W) bool, True means background/void/ignore


def _get_predictions_table_df(run: Run) -> pd.DataFrame:
    arts = list(run.logged_artifacts())
    candidates = [a for a in arts if "test_predictions_final" in a.name]
    if not candidates:
        raise RuntimeError(
            f"Could not find a 'test_predictions_final' artifact for run {run.path}."
        )
    artifact = run.use_artifact(candidates[-1])
    table = artifact.get("test_predictions_final_epoch")

    return table.get_dataframe()  # type: ignore


def _normalize_gt_pred_masks(
    ground_truth_hw: np.ndarray, prediction_hw: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns (gt_cls, pred_cls, ignore_mask), where gt_cls/pred_cls are in 0..17.

    We support two common formats:
    - PASTIS original GT: 0..19 (0=bkg, 19=void) -> ignore bkg/void and remap 1..18 -> 0..17.
    - Remapped GT used in this repo: 0..17 + ignore_index(=18) -> ignore gt==18 and keep 0..17.
    """
    gt = np.asarray(ground_truth_hw)
    pred = np.asarray(prediction_hw)

    if gt.min() == 0 and gt.max() == 19:
        ignore_mask = (gt == 0) | (gt == 19)
        gt_cls = gt.astype(np.int64) - 1
    elif gt.max() == 18:
        ignore_mask = gt == 18
        gt_cls = gt.astype(np.int64)
    else:
        ignore_mask = np.zeros_like(gt, dtype=bool)
        gt_cls = gt.astype(np.int64)

    # Prediction: often 0..17 (model classes), sometimes 1..18 (PASTIS-like)
    if pred.min() >= 1 and pred.max() <= 18:
        pred_cls = pred.astype(np.int64) - 1
    else:
        pred_cls = pred.astype(np.int64)

    return gt_cls, pred_cls, ignore_mask


def _compute_embedding_pca_rgb(
    embedding_hwd: np.ndarray, ignore_mask_hw: np.ndarray, seed: int = 0
) -> np.ndarray:
    emb = np.asarray(embedding_hwd, dtype=np.float32)
    h, w, d = emb.shape
    x = emb.reshape(-1, d)
    valid = (~ignore_mask_hw).reshape(-1)
    x_valid = x[valid]
    if x_valid.size == 0:
        x_valid = x
    n_fit = min(20_000, x_valid.shape[0])
    rng = np.random.default_rng(seed)
    fit_idx = rng.choice(x_valid.shape[0], size=n_fit, replace=False)
    pca = PCA(n_components=3, random_state=seed)
    pca.fit(x_valid[fit_idx])
    z = pca.transform(x).reshape(h, w, 3)

    # Robust scaling to [0,1] per channel for visualization
    rgb = np.empty_like(z, dtype=np.float32)
    for c in range(3):
        chan = z[..., c]
        lo, hi = np.nanpercentile(chan[~ignore_mask_hw], [2, 98])
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = float(chan.min()), float(chan.max())
        rgb[..., c] = np.clip((chan - lo) / (hi - lo + 1e-6), 0.0, 1.0)
    rgb[ignore_mask_hw] = np.nan
    return rgb


def plot_prediction_map(
    *runs: Run,
    patch_id: int | None = None,
    alpha: float = 1.0,
    pca_alpha: float = 1.0,
    seed: int = 0,
    basemap_source=cx.providers.Esri.WorldImagery,  # type: ignore
    basemap_attribution: str | None = "© Esri",
    basemap_attribution_size: int = 6,
    title_suffix: str = "",
    output_filename_prefix: str = "prediction_map",
) -> None:
    """
    For each run, adds a row with 3 panels (left->right):
    1) PCA (RGB) of the embedding over basemap
    2) Predicted segmentation over basemap
    3) Ground-truth segmentation over basemap

    Notes:
    - Predictions are expected in 0..17 (18 classes excluding background/void).
    - Pixels belonging to background/void are masked out based on the GT mask.
    - Label names are taken from `PASTIS_LABEL_NAMES` with mapping cls k->PASTIS k+1.
    - Basemap attribution is kept but shortened by default. Set `basemap_attribution=""`
      if you explicitly want no attribution text.
    - title_suffix: optional suffix to add to the figure title
    - output_filename_prefix: prefix for the output PDF filename
    """

    dfs = [_get_predictions_table_df(run) for run in runs]
    if patch_id is None:
        patch_id = int(dfs[0].iloc[0]["patch_id"])
    patch_id_int = int(patch_id)

    nrows = len(runs)
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=3,
        figsize=(18, 5.2 * nrows),
        squeeze=False,
        gridspec_kw={"wspace": 0.02, "hspace": 0.10},
    )

    # Discrete colormap for 18 classes (0..17). Masked pixels become transparent.
    base_colors = list(plt.get_cmap("tab20").colors[:18])  # type: ignore
    seg_cmap = ListedColormap(base_colors, name="pastis_18")
    seg_cmap.set_bad((0, 0, 0, 0))

    meta = dataset.pastis_dataset.meta_patch
    patch_geom = meta.loc[patch_id_int].geometry
    patch_geom_3857 = gpd.GeoSeries([patch_geom], crs=meta.crs).to_crs(epsg=3857)
    xmin, ymin, xmax, ymax = patch_geom_3857.total_bounds
    extent = (xmin, xmax, ymin, ymax)

    class_legend_handles: list[Patch] | None = None

    for row_i, (run, df) in enumerate(zip(runs, dfs, strict=True)):
        sub = df[df["patch_id"].astype(int) == patch_id_int]
        if len(sub) == 0:
            sub = df.iloc[[0]]
        r = sub.iloc[0]

        gt_cls, pred_cls, ignore_mask = _normalize_gt_pred_masks(
            r["ground_truth_mask"], r["prediction_mask"]
        )
        payload = _PredictionPayload(
            patch_id=int(r["patch_id"]),
            embedding_hwd=np.asarray(r["embedding"]),
            ground_truth_hw=gt_cls,
            prediction_hw=pred_cls,
            ignore_mask_hw=ignore_mask,
        )

        # Determine labels present (for a compact legend)
        gt_labels = sorted(
            int(x) for x in np.unique(payload.ground_truth_hw[~payload.ignore_mask_hw])
        )
        pred_labels = sorted(
            int(x) for x in np.unique(payload.prediction_hw[~payload.ignore_mask_hw])
        )
        labels_present = sorted(set(gt_labels) | set(pred_labels))
        if class_legend_handles is None:
            class_legend_handles = [
                Patch(
                    facecolor=seg_cmap(lbl),
                    edgecolor="none",
                    label=PASTIS_LABEL_NAMES[lbl + 1],
                )
                for lbl in labels_present
                if (lbl + 1) in PASTIS_LABEL_NAMES
            ]

        # --- PCA panel (now first) ---
        ax_pca = axes[row_i, 0]
        ax_pca.set_xlim(xmin, xmax)
        ax_pca.set_ylim(ymin, ymax)
        cx.add_basemap(
            ax_pca,
            source=basemap_source,
            attribution=basemap_attribution,
            attribution_size=basemap_attribution_size,
            reset_extent=False,
        )
        rgb = _compute_embedding_pca_rgb(
            payload.embedding_hwd, payload.ignore_mask_hw, seed=seed
        )
        ax_pca.imshow(rgb, extent=extent, origin="upper", alpha=pca_alpha)
        ax_pca.set_axis_off()

        # --- Prediction panel (now second) ---
        ax_pred = axes[row_i, 1]
        ax_pred.set_xlim(xmin, xmax)
        ax_pred.set_ylim(ymin, ymax)
        cx.add_basemap(
            ax_pred,
            source=basemap_source,
            attribution=basemap_attribution,
            attribution_size=basemap_attribution_size,
            reset_extent=False,
        )
        pred_masked = np.ma.masked_where(payload.ignore_mask_hw, payload.prediction_hw)
        ax_pred.imshow(
            pred_masked,
            extent=extent,
            origin="upper",
            cmap=seg_cmap,
            vmin=0,
            vmax=17,
            alpha=alpha,
            interpolation="nearest",
        )
        ax_pred.set_axis_off()

        # --- Ground truth panel (now third) ---
        ax_gt = axes[row_i, 2]
        ax_gt.set_xlim(xmin, xmax)
        ax_gt.set_ylim(ymin, ymax)
        cx.add_basemap(
            ax_gt,
            source=basemap_source,
            attribution=basemap_attribution,
            attribution_size=basemap_attribution_size,
            reset_extent=False,
        )
        gt_masked = np.ma.masked_where(payload.ignore_mask_hw, payload.ground_truth_hw)
        ax_gt.imshow(
            gt_masked,
            extent=extent,
            origin="upper",
            cmap=seg_cmap,
            vmin=0,
            vmax=17,
            alpha=alpha,
            interpolation="nearest",
        )
        ax_gt.set_axis_off()

    # Column headers (once, on the top row)
    axes[0, 0].set_title("Embedding PCA", fontsize=12, pad=20)
    axes[0, 1].set_title("Predicted label", fontsize=12, pad=20)
    axes[0, 2].set_title("Ground truth", fontsize=12, pad=20)

    # Global title centered with more space above
    title = f"Patch {patch_id_int}"
    if title_suffix:
        title += f" {title_suffix}"
    fig.suptitle(title, y=0.995, fontsize=14, ha="center")
    fig.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.12)

    # Create combined legend with PCA and classes
    legend_handles_all = []

    # Add PCA legend items (no spacer to avoid alignment issues)
    pca_legend_handles = [
        Patch(facecolor="red", edgecolor="none", label="PC1 (Red)"),
        Patch(facecolor="green", edgecolor="none", label="PC2 (Green)"),
        Patch(facecolor="blue", edgecolor="none", label="PC3 (Blue)"),
    ]

    # Combine PCA legend + class legend
    if class_legend_handles:
        legend_handles_all = pca_legend_handles + class_legend_handles

        # Arrange in rows to fill the width better
        ncol = 7  # This should put items in a nice grid

        fig.legend(
            handles=legend_handles_all,
            loc="lower center",
            bbox_to_anchor=(0.5, 0.01),
            frameon=False,
            fontsize="small",
            ncol=ncol,
        )

    # Row titles: run name at the bottom of each row
    fig.canvas.draw()
    for row_i, run in enumerate(runs):
        row_axes = axes[row_i, :]
        left = row_axes[0].get_position().x0
        right = row_axes[-1].get_position().x1
        bottom = min(ax.get_position().y0 for ax in row_axes)
        fig.text(
            (left + right) / 2,
            bottom - 0.012,
            str(run.name),
            ha="center",
            va="top",
            fontsize=11,
        )

    plt.show()
    fig.savefig(
        os.path.join(output_dir, f"{output_filename_prefix}_patch_{patch_id_int}.pdf"),
        backend="pgf",
    )

In [ ]:
plot_prediction_map(*preview_runs)


### Prediction maps for scarce data (30 patches per fold)


In [ ]:
plot_prediction_map(
    *preview_runs_scarce,
    title_suffix="(Scarce data: 30 patches/fold)",
    output_filename_prefix="prediction_map_scarce",
)